# 02 - Mean car prices by manufacturer (1985 automobile market)

**Question.** How much do the mean prices of car manufacturers differ, and how skewed is the
price distribution?

**Data.** `data/automobiles.csv` - UCI Machine Learning Repository *Automobile* dataset (201
cars, 14 attributes). Licence: CC BY 4.0.


In [1]:
import pandas as pd
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, LinearColorMapper
from bokeh.plotting import figure, show
from bokeh.transform import linear_cmap

output_notebook()
dataset = pd.read_csv('../data/automobiles.csv')
print("cars:", dataset.shape[0], "| makes:", dataset['make'].nunique())
dataset.head()


Loading BokehJS ...

cars: 201 | makes: 22


,make,fuel-type,num-of-doors,body-style,engine-location,length,width,height,num-of-cylinders,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,alfa-romero,gas,two,convertible,front,168.8,64.1,48.8,four,111,5000,21,27,13495
1,alfa-romero,gas,two,convertible,front,168.8,64.1,48.8,four,111,5000,21,27,16500
2,alfa-romero,gas,two,hatchback,front,171.2,65.5,52.4,six,154,5000,19,26,16500
3,audi,gas,four,sedan,front,176.6,66.2,54.3,four,102,5500,24,30,13950
4,audi,gas,four,sedan,front,176.6,66.4,54.3,five,115,5500,18,22,17450


## Plot 1 - the price of every car

One marker per car, sorted by the original dataset index.


In [2]:
dataset['index'] = dataset.index
plot = figure(title='Car prices (201 cars)', x_axis_label='Car index',
              y_axis_label='Price (USD)', width=900, height=400)
plot.scatter(dataset['index'], dataset['price'], size=5, marker='circle')
show(plot)
print("median:", dataset['price'].median(), "| max:", dataset['price'].max(),
      "| cars over 30,000:", int((dataset['price'] > 30000).sum()))


median: 10295.0 | max: 45400 | cars over 30,000: 14


## Plot 2 - the mean price per manufacturer

The means are drawn as bars; the label orientation is vertical so the 22 names stay readable.


In [3]:
grouped = dataset.groupby('make', as_index=False).mean(numeric_only=True)
grouped = grouped.sort_values('price', ascending=False)
print(grouped[['make', 'price']].head(5).to_string(index=False))
print("...")
print(grouped[['make', 'price']].tail(3).to_string(index=False))

bar = figure(title='Mean car price by manufacturer (sorted)',
             x_axis_label='Manufacturer', y_axis_label='Mean price (USD)',
             x_range=list(grouped['make']), width=900, height=440)
bar.vbar(x=grouped['make'], top=grouped['price'], width=0.7)
bar.xaxis.major_label_orientation = 'vertical'
show(bar)


         make        price
       jaguar 34600.000000
mercedes-benz 33647.000000
      porsche 31400.500000
          bmw 26118.750000
        volvo 18063.181818
...
     make       price
 plymouth 7963.428571
    dodge 7875.444444
chevrolet 6007.000000


## Plot 3 - the same means, colour-encoded by price

A `LinearColorMapper` on the price column with the `Magma256` palette: dark = cheap, pale = expensive.


In [4]:
mapper = LinearColorMapper(palette='Magma256', low=grouped['price'].min(),
                           high=grouped['price'].max())

coloured = figure(title='Mean car price by manufacturer (colour = mean price)',
                  x_axis_label='Manufacturer', y_axis_label='Mean price (USD)',
                  x_range=list(grouped['make']), width=900, height=440)
coloured.scatter('make', 'price', source=ColumnDataSource(grouped), size=16,
                 marker='circle', line_color='white',
                 color=linear_cmap('price', 'Magma256', grouped['price'].min(),
                                   grouped['price'].max()))
coloured.xaxis.major_label_orientation = 'vertical'
show(coloured)


**Result.** The mean price per manufacturer ranges from **6,007 USD** (Chevrolet) to **34,600
USD** (Jaguar), a factor of **5.8x**, while the median car costs 10,295 USD and 14 of the 201
cars cost more than 30,000 USD. The distribution is strongly right-skewed: a handful of luxury
manufacturers pull the mean far above the median, which is exactly why the sorted bars and the
colour encoding make the ranking easier to read than the raw scatter.

**Limitation.** The prices are 1985 USD observations without inflation adjustment, and the
dataset is a small convenience sample, so the means describe this sample rather than the market.
